# Reinforcement learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/11_reinforcement_learning.ipynb)

Official API intro to reinforcement learning as a *planner*: a `StochasticPlanningProblem` states the task (plant, cost, box, exit rule, and what is random), `MonteCarloEvaluator` scores any controller on it, and `ReinforcementLearningPlanner` learns a neural feedback law $u = \pi_\theta(x)$ that comes back as an ordinary controller block. In between sit the pieces the planner is made of: the rollout environment the learner sees, the one discount it trains with, and an update rule chosen from two families, on-policy PPO and off-policy SAC. Because the learned law is a `System`, the closed loop compiles, linearizes and differentiates like everything else in minilink; the last section shows that off.

**Scripts for depth:** `examples/demos/rl/` (pendulum, PPO against SAC, cart-pole, car on a circuit, rocket landing, drone)

**Teaching notebooks:** [`policy_gradient_to_ppo`](../teaching/reinforcement_learning/policy_gradient_to_ppo.ipynb) (the mathematics behind these functions) · [`gymnasium_interface`](../teaching/reinforcement_learning/gymnasium_interface.ipynb) (the same task as a Gymnasium environment) · [`pendulum_swing_up_vi_vs_lqr_vs_rl`](../teaching/reinforcement_learning/pendulum_swing_up_vi_vs_lqr_vs_rl.ipynb) · [`drone_learn_to_fly`](../teaching/reinforcement_learning/drone_learn_to_fly.ipynb) · [`experimental/rl/RL_README.md`](../experimental/rl/RL_README.md) (tuning lessons)

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## The plant and the cost

A torque-limited pendulum ($\theta = 0$ hanging, $\theta = \pi$ upright) with less torque than gravity, so reaching the top takes a pumping motion. The cost is the textbook pair $J = \int g\,dt + h(x_f)$; the running cost is periodic in the angle, zero upright and two hanging. It is written with `jax.numpy` so it traces inside the compiled rollouts.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from minilink import CostFunction, DiagramSystem, Pendulum, PendulumWithNoisePort
from minilink.analysis import bode, plot_pzmap
from minilink.control import NeuralPolicyController, StateFeedbackController, angle_features
from minilink.planning import (
    MonteCarloEvaluator,
    PlanningProblem,
    ReinforcementLearningPlanner,
    StochasticPlanningProblem,
    Uniform,
    as_stochastic,
)

TORQUE = 4.0  # Nm, below m g l = 9.81 Nm
DT = 0.05  # control period of the learned law


def torque_limited_pendulum(cls=Pendulum):
    plant = cls()
    plant.inputs["u"].lower_bound = np.array([-TORQUE])
    plant.inputs["u"].upper_bound = np.array([TORQUE])
    plant.state.lower_bound = np.array([-4 * np.pi, -20.0])  # the training box
    plant.state.upper_bound = np.array([4 * np.pi, 20.0])
    return plant


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        theta, dtheta = x
        return (1.0 + jnp.cos(theta)) + 0.01 * dtheta**2 + 0.01 * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


plant = torque_limited_pendulum()
cost = SwingUpCost()
X_UP = np.array([np.pi, 0.0])

## A stochastic planning problem

`PlanningProblem` is deterministic: one start, one plant. `StochasticPlanningProblem` keeps the same spine and adds what is random — here the initial state, drawn uniformly over the whole circle with small rates — and the criterion (the expected cost). Two more declarations settle what episodes mean:

- `tf = inf` makes it an infinite-horizon task; the planners then pick a discount (or read `cost.discount_rate`) and an episode length.
- The state bounds are the allowed box $X$. Leaving it is a constraint violation whose price is declared **on the problem** (`on_exit`, `exit_cost`), never hidden in the cost function, so trajectory optimization, dynamic programming and reinforcement learning score the same trajectory the same way. Here we keep the default (`"infeasible"`, no price). For reinforcement learning that means an episode leaving the box is *truncated*: it ends, and the critic's estimate of the exit state's value stands in for the rest of the return. That is the Gymnasium convention, and an approximation a policy can exploit when leaving is cheap; pricing the exit (`on_exit="terminate"`, `exit_cost=...`) removes the bootstrap. The planner's environment states which rule is in force.

The same object answers `sample_x0` for Monte Carlo and `nominal()` for the deterministic planners.

In [ ]:
problem = StochasticPlanningProblem(
    plant,
    cost=cost,
    tf=np.inf,
    x0_distribution=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
)
print("horizon:", problem.horizon_kind(), "| exit rule:", problem.on_exit, "| x_start =", problem.x_start)
print("three draws of x0:\n", np.round(problem.sample_x0(0, n=3), 2))
print("nominal problem:", type(problem.nominal()).__name__, "from", problem.nominal().x_start)

**A deterministic problem is the one-start case.** `as_stochastic` turns a `PlanningProblem`, the object of the dynamic programming and trajectory optimization chapters, into the stochastic problem whose every draw is its single `x_start`, with nothing else random; `nominal()` is the bridge back. The planner applies it on its own when handed a deterministic problem, so that problem trains as it is, from one start only, which covers the state space poorly: the start distribution above is what teaches the law to swing up from anywhere.

In [ ]:
one_start = as_stochastic(PlanningProblem(plant, x_start=[0.05, 0.0], cost=cost, tf=np.inf))
print(type(one_start).__name__, "| every draw of x0 is x_start:\n", one_start.sample_x0(0, n=3))
print("and back:", type(one_start.nominal()).__name__)

## Monte Carlo evaluation of any controller

`MonteCarloEvaluator` closes the loop with *any* state-feedback block, draws starts from the problem, and reports the distribution of $J$: mean, spread, worst case, and the failure rate (trials that left the box). It is the same yardstick for a hand-tuned law and for a learned one.

As a baseline, a PD law about the upright equilibrium, $u = -K (x - x_{up})$, with the torque clipped to the actuator limit. It balances the pendulum when it starts near the top and can do nothing from below.

In [ ]:
pd_ctl = StateFeedbackController(K=[[30.0, 8.0]], xbar=X_UP)
evaluator = MonteCarloEvaluator(problem, dt=DT, n_trials=100, seed=1)

report_pd = evaluator.evaluate(pd_ctl)
print("PD law:", report_pd)
print("best 10 trials (started near the top):", np.round(np.sort(report_pd.J)[:10], 2))

## What the learner sees: the rollout environment

`ReinforcementLearningPlanner` reads the problem the way `DynamicProgrammingPlanner` does, compiles the plant with the JAX backend, and wraps problem and plant into `planner.env`. Its `step` is one control period of the task:

- the dynamics $x_{k+1} = f(x_k, u_k, w_k)$ with the input held over the period;
- the reward $r_k = -g(x_k, u_k)\,\Delta t$, the running cost with its sign flipped, so maximizing the return minimizes $J$;
- two ways to end an episode: *terminated*, when its cost is fully counted (the finite horizon is reached, or a priced exit), or *truncated*, when the value of the next state is still to come (the episode length is reached, or an unpriced exit).

Every step is a pure JAX function, so many plants run in parallel inside one `lax.scan`, and the policy is updated with the chosen algorithm. The policy sees *features* of the state — here the angle as $(\cos\theta, \sin\theta)$ and a scaled rate, via `angle_features` — but the law is still $u = \pi(x)$.

In [ ]:
planner = ReinforcementLearningPlanner(
    problem,
    dt=DT,
    features=angle_features(angles=[0], scales={1: 0.1}),
    hidden=(32, 32),
    algorithm="ppo",
    n_envs=64,
    n_steps=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
    verbose=0,
)
env = planner.env
print(env.describe())

# One control period by hand: full torque from almost hanging
x, u = jnp.array([0.05, 0.0]), jnp.array([TORQUE])
x_next, t_next, r, terminated, truncated = env.step(x, 0.0, u, jax.random.PRNGKey(0))
print("x_next =", np.round(np.asarray(x_next), 4), "at t =", float(t_next))
print(f"r = {float(r):.5f}, and -g(x, u) dt = {-float(cost.g(x, u)) * DT:.5f}")
print("terminated:", bool(terminated), "| truncated:", bool(truncated))

## The discount

An infinite-horizon cost never ends, so learning discounts it: a factor $\gamma$ per control period, which weighs the future over a horizon of about $\Delta t / (-\ln\gamma)$ seconds. The discount has one owner, the algorithm, and is filled in this order:

1. a value set on the algorithm object, `PPO(gamma=...)`;
2. the planner's `gamma=`;
3. a rate $\rho$ declared on the cost, `discount_rate`, giving $\gamma = e^{-\rho\,\Delta t}$;
4. otherwise 0.99, with a warning that names the horizon it implies.

Monte Carlo evaluation always scores the task's own cost. The planner above chose 0.97, a horizon of about 1.6 s: long enough to plan one swing, short enough to keep the value estimates steady. A planner with nothing declared announces its default.

In [ ]:
print(f"planner.gamma = {planner.gamma}: a horizon of {DT / -np.log(planner.gamma):.1f} s")
default = ReinforcementLearningPlanner(problem, dt=DT, hidden=(8,), verbose=0)  # nothing declared
print(f"default gamma = {default.gamma}: a horizon of {DT / -np.log(default.gamma):.1f} s")

## Learning, and the evaluation behind the plan

`solve` trains for a budget of plant steps, then scores the learned law on the problem by Monte Carlo. The plan carries the headline numbers; `planner.last_evaluation` keeps the whole report, trial by trial.

In [ ]:
plan = planner.solve(timesteps=120_000)
print(plan.metadata.message, f"in {plan.metadata.solve_time_s:.1f} s")
report = planner.last_evaluation
print("the five worst of", report.J.size, "trials:", np.round(np.sort(report.J)[-5:], 2))
planner.plot_learning_curve()
plt.show()

## The learned law is a controller block

`get_controller()` returns a `NeuralPolicyController`: features, a multilayer perceptron whose weights live in `params["mlp"]`, and a map onto the actuator bounds. It draws its law with `plot_control_law` like LQR or value iteration, scores on the same Monte Carlo yardstick, and closes the loop with `@`.

`planner.nominal_trajectory()` is the law as the planner sees it: from the task's nominal start, the input held over each control period, nominal parameters and disturbances. Closing the loop with `@` and simulating is the continuous-time view of the same law.

In [ ]:
rl_ctl = planner.get_controller()
rl_ctl.plot_control_law(x_axis=0, y_axis=1, u_axis=0)  # torque vs (theta, dtheta)

report_rl = evaluator.evaluate(rl_ctl)
print("PD law:", report_pd)
print("RL law:", report_rl)

In [ ]:
nominal = planner.nominal_trajectory()  # from problem.x_start, on the training grid
plant.plot_trajectory(nominal)

In [ ]:
plant.x0 = np.array([0.05, 0.0])  # hanging, a tiny tip to break the symmetry
cl_sys = rl_ctl @ plant
cl_sys.name = "Pendulum with the learned law"
cl_sys.plot_diagram()
traj = cl_sys.compute_trajectory(tf=10.0, dt=0.01)
cl_sys.plot_trajectory(traj)
cl_sys.animate(traj)

## The other family: off-policy SAC

`algorithm="sac"` swaps the update rule and keeps everything else: the same problem, environment, evaluator and controller block. Soft Actor-Critic stores transitions in a replay buffer and takes a gradient step for every plant step, with a tanh-squashed Gaussian law, twin action-value critics and a learned temperature that keeps exploring. Here it needs about a quarter of PPO's plant steps and many times its wall time: sample efficiency against computation, the trade between the two families.

In [ ]:
sac = ReinforcementLearningPlanner(
    problem,
    dt=DT,
    features=angle_features(angles=[0], scales={1: 0.1}),
    hidden=(64, 64),
    algorithm="sac",
    n_envs=8,
    n_steps=16,
    gamma=0.98,
    learning_starts=2000,
    verbose=0,
)
sac_plan = sac.solve(timesteps=30_000)
print(sac_plan.metadata.message, f"in {sac_plan.metadata.solve_time_s:.1f} s")
print("PPO law:", evaluator.evaluate(rl_ctl))
print("SAC law:", evaluator.evaluate(sac.get_controller()))

## Everything is a System: compile, linearize, differentiate

The closed loop `rl_ctl @ plant` is a diagram like any other, so the analysis tools apply to the *learned* law with no adapter. Linearizing at the upright equilibrium differentiates through the network and the plant in one JAX trace; the eigenvalues of $A$ tell whether the neural law stabilizes the top.

In [ ]:
lin = cl_sys.linearize(X_UP)  # exact Jacobians by autodiff through pi(x) and f(x, u)
poles = np.linalg.eigvals(lin.A())
print("closed-loop A at the top:\n", np.round(lin.A(), 3))
print("closed-loop poles:", np.round(poles, 3), "-> stable" if np.all(poles.real < 0) else "-> unstable")

**Disturbance rejection of a neural controller.** To ask a frequency-domain question we need an input to ask it from: the catalog `PendulumWithNoisePort` has a disturbance torque port `w`. Wire the same learned block around it by hand, expose `w` as the diagram's boundary input, and the frequency tools see one more `System`. Below: the poles of the open-loop plant (a real unstable pole at the top) against those of the closed loop with the RL law, the disturbance-to-angle Bode plots of both (the magnitudes are close because the learned law is not stiff at 4 Nm, the phases differ by the missing unstable pole), and the closed-loop pole-zero map.

In [ ]:
plant_w = torque_limited_pendulum(PendulumWithNoisePort)

loop = DiagramSystem()
loop.name = "RL law around the pendulum with a disturbance torque"
loop.add_subsystem(rl_ctl, "ctl")
loop.add_subsystem(plant_w, "plant")
loop.connect("plant", "y", "ctl", "x")
loop.connect("ctl", "u", "plant", "u")
loop.add_input_port("w", dim=1)
loop.connect("input", "w", "plant", "w")
loop.connect_new_output_port("plant", "y", "y")
loop.plot_diagram()

print("open-loop poles at the top:  ", np.round(np.linalg.eigvals(plant_w.linearize(X_UP).A()), 2))
print("closed-loop poles at the top:", np.round(np.linalg.eigvals(loop.linearize(X_UP).A()), 2))

w, mag_open, phase_open = bode(plant_w, X_UP, of=("y", 0), wrt="w")
_, mag_closed, phase_closed = bode(loop, X_UP, of=("y", 0), wrt="w")
fig, (ax_mag, ax_phase) = plt.subplots(2, 1, sharex=True, figsize=(7, 5))
ax_mag.semilogx(w, mag_open, label="open loop: w -> theta (unstable)")
ax_mag.semilogx(w, mag_closed, label="closed loop with the RL law")
ax_mag.set_ylabel("|G| [dB]")
ax_mag.legend()
ax_phase.semilogx(w, phase_open)
ax_phase.semilogx(w, phase_closed)
ax_phase.set_ylabel("phase [deg]")
ax_phase.set_xlabel("frequency [rad/s]")
for ax in (ax_mag, ax_phase):
    ax.grid(True, which="both", alpha=0.3)
plt.show()

plot_pzmap(loop, X_UP, of=("y", 0), wrt="w")

**Gradients through the physics.** The compiled diagram exposes the parametric step `x_{k+1} = \text{rk4}(x_k, u_k, t_k, \Delta t; p)` with `p` holding the controller weights *and* the plant parameters. A closed-loop rollout written as a `lax.scan` is then a function of both, and `jax.grad` gives, in one call, the sensitivity of the cost to the pendulum's mass and the gradient with respect to every network weight. A short line search along that gradient (the step size that lowers the cost most among a few candidates) fine-tunes the law learned by sampling by backpropagating through the dynamics.

In [ ]:
evaluator_jax = cl_sys.compile(backend="jax")
u_none = jnp.zeros(0)  # the closed loop has no boundary input
N = int(10.0 / DT)


def rollout_cost(params):
    def step(carry, k):
        x, t = carry
        u = rl_ctl.action(x, params["ctl"])  # the applied torque, for the cost
        J_k = cost.g(x, u, t) * DT
        x_next = evaluator_jax.rk4_step_trace_p(x, u_none, t, DT, params)
        return (x_next, t + DT), J_k

    (_, _), J_k = jax.lax.scan(step, (jnp.array([0.05, 0.0]), 0.0), jnp.arange(N))
    return jnp.sum(J_k)


params = jax.tree_util.tree_map(jnp.asarray, {"ctl": rl_ctl.params, "sys": plant.params})
J0, grads = jax.jit(jax.value_and_grad(rollout_cost))(params)
print(f"J of the swing-up from hanging: {float(J0):.3f}")
print("dJ/dm (pendulum mass):", float(grads["sys"]["m"]), "| dJ/dl (length):", float(grads["sys"]["l"]))
n_weights = sum(int(np.prod(g.shape)) for g in jax.tree_util.tree_leaves(grads["ctl"]))
print(f"gradient with respect to all {n_weights} policy weights: norm {float(jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree_util.tree_leaves(grads['ctl']))) ):.3f}")

# One gradient step on the weights, through the plant: a line search on the step size
rollout_cost_jit = jax.jit(rollout_cost)
best_lr, best_J = 0.0, float(J0)
for lr in (0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001):
    tuned = dict(params)
    tuned["ctl"] = jax.tree_util.tree_map(lambda w, g: w - lr * g, params["ctl"], grads["ctl"])
    J_lr = float(rollout_cost_jit(tuned))
    print(f"  step {lr:6.3f}: J = {J_lr:.3f}")
    if J_lr < best_J:
        best_lr, best_J = lr, J_lr
print(f"best step {best_lr}: J {float(J0):.3f} -> {best_J:.3f} ({100 * (best_J - float(J0)) / float(J0):+.1f}%)")

## What to read next

- [`teaching/reinforcement_learning/policy_gradient_to_ppo.ipynb`](../teaching/reinforcement_learning/policy_gradient_to_ppo.ipynb): the mathematics — the policy gradient, the baseline, actor-critic, generalized advantage estimation and PPO — each checked against the functions used above.
- [`teaching/reinforcement_learning/gymnasium_interface.ipynb`](../teaching/reinforcement_learning/gymnasium_interface.ipynb): the same task as a Gymnasium environment, the interface the rest of the RL ecosystem speaks.
- [`teaching/reinforcement_learning/pendulum_swing_up_vi_vs_lqr_vs_rl.ipynb`](../teaching/reinforcement_learning/pendulum_swing_up_vi_vs_lqr_vs_rl.ipynb): value iteration, LQR and PPO on one Monte Carlo yardstick.
- `examples/demos/rl/`: the official scripts, each a `StochasticPlanningProblem` solved by the planner and scored by Monte Carlo, with PPO against SAC side by side in `pendulum_ppo_vs_sac_rl.py`.
- `experimental/rl/RL_README.md`: what it took to make each task learn — the state-box exit exploit, periodic features, action normalization, reward scale, discount versus the task's time scale.
- `docs/plans/rl-planner-vision.md`: the design — one problem description, two verbs (`solve`, `evaluate`), one planner hosting an algorithm family.